In [1]:
import pytest
from dptb.data.build import build_dataset
from dptb.data.dataset import DefaultDataset
from dptb.data.transforms import OrbitalMapper
from dptb.data import AtomicDataset, DataLoader, AtomicDataDict,AtomicData
from dptb.nn.nnsk import NNSK
from dptb.utils.torch_geometric import Batch
from dptb.nn.sktb.socbasic import get_soc_matrix_cubic_basis
import matplotlib.pyplot as plt
from dptb.utils.constants import Harte2eV, Bohr2Ang
import torch

In [2]:
# 过一遍代码内部的数据流
# 构建dataset
root_directory='.'
set_options = {
    "r_max": 6.0,
    "root": "./data",
    "prefix": "kpath",
    "get_eigenvalues": True,
    "get_Hamiltonian": False,
}
common_options={"basis": {"B": ["2s", "2p"], "N": ["2s", "2p"]}}
dataset = build_dataset(**set_options, **common_options)

# call data loader 拿到一个batch
dload= DataLoader(dataset, batch_size=1, shuffle=False)
batch = next(iter(dload))
batch = batch.to_dict()

/opt/mamba/envs/unisk-dev/lib/python3.10/site-packages/torch/nested/__init__.py:107: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return torch._nested_tensor_from_tensor_list(ts, dtype, None, device, None)
The cutoffs in data and model are not checked. be careful!


In [3]:
# 仿照代码内部对batch data 初始化操作
idp_sk = OrbitalMapper(basis={"B": ["2s", "2p"], "N": ["2s", "2p"]},method="sktb")
batch = AtomicDataDict.with_edge_vectors(batch, with_lengths=True)
idp_sk(batch)
data = batch

In [4]:
data[AtomicDataDict.EDGE_LENGTH_KEY]
edge_index = data[AtomicDataDict.EDGE_TYPE_KEY].flatten()
edge_number = idp_sk.untransform_bond(edge_index).T
edge_atom_types = idp_sk.transform_atom(edge_number.flatten()).reshape(2, -1)

In [5]:
# 对接dftbsk models， 需要加载对应的skf文件
from dptb.nn.dftb.sk_param import SKParam
sk_path = "./slakos"
basis = {'B':['2s','2p'],"N":["2s","2p"]}
skp = SKParam(basis=common_options['basis'],skdata=sk_path)

In [6]:
skp.skdict['Highest_Occu_U'].flatten()

tensor([10.3525, 13.3009])

In [7]:
Highest_Occu_U = skp.skdict['Highest_Occu_U'].flatten()
err_index = torch.where(Highest_Occu_U < 1e-6)[0]
if len(err_index) > 0:
    print("Failure in short-range gamma, U too small for atom number : ", idp_sk.untransform_atom(err_index))

In [8]:
edge_atom_types

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0,
         0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0],
        [1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1,
         0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0,
         0, 0, 0, 0, 0, 0]])

In [9]:
Ua = skp.skdict['Highest_Occu_U'][edge_atom_types[0]].reshape(-1)
Ub = skp.skdict['Highest_Occu_U'][edge_atom_types[1]].reshape(-1)

In [10]:
# 注意 data[AtomicDataDict.EDGE_LENGTH_KEY] edge 在代码中指的是 r_ab >0 的边。不包含 rab=0的特殊情况（节点） 
assert (data[AtomicDataDict.EDGE_LENGTH_KEY] > 1e-6).all(), "Failure in short-range gamma, r_ab negative"

In [11]:
# 先计算边的gamma
expGamma = torch.zeros([edge_index.shape[0]], dtype=torch.float32)
# equ_u_mask = edge_atom_types[0] == edge_atom_types[1]
# # 相同类型的原子 u相同，但是不同不同原子是否能可能出现u很接近呢？
equ_u_mask = torch.abs(Ua - Ub)  < 1e-6

In [12]:
from scipy import constants
# The Coulomb constant in SI units
k_e = 1/(4*torch.pi*constants.epsilon_0)
# k_e * q^2/r # is the energy in Joules, we convert it to eV 
# and in the follwing code we will use:
#  q, the charge of the electron unit if e. 
#  r, the distance, unit of Angstrom.
#  E, energy, unit of eV.
#  E = k_e * q^2 / r  <=>  factor q^2 / r (here q is a numer, and r is in unit of Angstrom, E is in eV)
R2E_factor = k_e * constants.e**2 / 10**-10 / constants.e # 1/R -> E, unit of eV
# then R unit to -> 1/eV, the factor should be 1/R2E_factor


In [13]:
tauMean = 0.5  * 16./5. * (Ua[equ_u_mask] + Ub[equ_u_mask]) # tau is unit of eV
rab_equU = 1./R2E_factor * data[AtomicDataDict.EDGE_LENGTH_KEY][equ_u_mask]
#这里注意量纲的转换，文章里面的公式定义是在原子单位制下的，因此 能量和 1/rab 的单位量纲才是对齐的
expGamma[equ_u_mask] = torch.exp(-tauMean * rab_equU) * (48.0 / rab_equU + 33.0 * tauMean +
                                                        9.0 * rab_equU * (tauMean ** 2) +
                                                        1.0 * (rab_equU ** 2) * (tauMean ** 3))/48.0

In [14]:
tauA = 16./5. * Ua[~equ_u_mask] 
tauB = 16./5. * Ub[~equ_u_mask]
rab_nequU = 1./R2E_factor * data[AtomicDataDict.EDGE_LENGTH_KEY][~equ_u_mask]

part1 = torch.exp(-tauA * rab_nequU) * (0.5 * tauB ** 4 * tauA / (tauA ** 2 - tauB ** 2) ** 2 - 
                                            (tauB ** 6 - 3.0 * tauB ** 4 * tauA ** 2) / (rab_nequU * (tauA ** 2 - tauB ** 2) ** 3))
part2 = torch.exp(-tauB * rab_nequU) * (0.5 * tauA ** 4 * tauB / (tauA ** 2 - tauB ** 2) ** 2 -
                                            (tauA ** 6 - 3.0 * tauA ** 4 * tauB ** 2) / (rab_nequU * (tauB ** 2 - tauA ** 2) ** 3))

expGamma[~equ_u_mask] = (part1 + part2)

In [15]:
expGamma_onsite = 1 * skp.skdict['Highest_Occu_U'][data[AtomicDataDict.ATOM_TYPE_KEY].flatten()].reshape(-1)

In [16]:
expGamma_onsite

tensor([13.3009, 10.3525])

In [17]:
from dptb.nn.dftb.gamma import calculate_expgamma


In [18]:
expGamma2, expGamma_onsite2 = calculate_expgamma(highest_occu_Us=skp.skdict['Highest_Occu_U'], 
                    edge_lengths=data[AtomicDataDict.EDGE_LENGTH_KEY],
                    edge_atom_types=edge_atom_types,
                    node_atom_types=data[AtomicDataDict.ATOM_TYPE_KEY])

In [34]:
abs(expGamma - expGamma2).max()

tensor(2.2650e-06)

In [20]:
expGamma_onsite - expGamma_onsite2

tensor([0., 0.])

In [21]:
expGammaau = torch.zeros([edge_index.shape[0]], dtype=torch.float32)

tauMean = 0.5  * 16./5. * (Ua[equ_u_mask] + Ub[equ_u_mask]) / Harte2eV # tau is unit of eV
rab_equU = data[AtomicDataDict.EDGE_LENGTH_KEY][equ_u_mask] / Bohr2Ang
#这里注意量纲的转换，文章里面的公式定义是在原子单位制下的，因此 能量和 1/rab 的单位量纲才是对齐的
expGammaau[equ_u_mask] = 1/rab_equU - torch.exp(-tauMean * rab_equU) * (48.0 / rab_equU + 33.0 * tauMean +
                                                        9.0 * rab_equU * (tauMean ** 2) +
                                                        1.0 * (rab_equU ** 2) * (tauMean ** 3))/48.0

tauA = 16./5. * Ua[~equ_u_mask] / Harte2eV
tauB = 16./5. * Ub[~equ_u_mask] / Harte2eV
rab_nequU =  data[AtomicDataDict.EDGE_LENGTH_KEY][~equ_u_mask] / Bohr2Ang

part1 = torch.exp(-tauA * rab_nequU) * (0.5 * tauB ** 4 * tauA / (tauA ** 2 - tauB ** 2) ** 2 - 
                                            (tauB ** 6 - 3.0 * tauB ** 4 * tauA ** 2) / (rab_nequU * (tauA ** 2 - tauB ** 2) ** 3))
part2 = torch.exp(-tauB * rab_nequU) * (0.5 * tauA ** 4 * tauB / (tauA ** 2 - tauB ** 2) ** 2 -
                                            (tauA ** 6 - 3.0 * tauA ** 4 * tauB ** 2) / (rab_nequU * (tauB ** 2 - tauA ** 2) ** 3))

expGammaau[~equ_u_mask] = 1/rab_nequU - (part1 + part2)

In [22]:
expGammaau * Harte2eV

tensor([2.8752, 2.7621, 3.3194, 2.7621, 2.8752, 3.7549, 2.7621, 2.7621, 3.7549,
        2.4900, 3.3194, 3.7549, 5.6636, 4.9047, 2.7621, 5.6636, 3.7549, 3.3194,
        2.4900, 8.2145, 4.9047, 2.7621, 4.9047, 2.8752, 3.7549, 5.6636, 8.2145,
        8.2145, 3.7549, 2.4900, 5.4760, 2.8735, 3.3129, 5.4760, 5.4760, 3.3129,
        2.8735, 3.3129, 2.8735, 2.8752, 2.7621, 3.3194, 2.7621, 2.8752, 3.7549,
        2.7621, 2.7621, 3.7549, 2.4900, 3.3194, 3.7549, 5.6636, 4.9047, 2.7621,
        5.6636, 3.7549, 3.3194, 2.4900, 8.2145, 4.9047, 2.7621, 4.9047, 2.8752,
        3.7549, 5.6636, 8.2145, 8.2145, 3.7549, 2.4900, 5.4760, 2.8735, 3.3129,
        5.4760, 5.4760, 3.3129, 2.8735, 3.3129, 2.8735])

In [23]:
expGamma

tensor([1.2839e-04, 4.5020e-04, 7.5969e-04, 4.5020e-04, 1.2839e-04, 9.7802e-03,
        4.5020e-04, 4.5020e-04, 9.7802e-03, 1.2562e-04, 7.5969e-04, 9.7802e-03,
        8.7077e-02, 7.5477e-02, 4.5020e-04, 8.7077e-02, 9.7802e-03, 7.5969e-04,
        1.2562e-04, 1.7459e+00, 7.5478e-02, 4.5020e-04, 7.5477e-02, 1.2839e-04,
        9.7802e-03, 8.7077e-02, 1.7459e+00, 1.7459e+00, 9.7802e-03, 1.2562e-04,
        2.7461e-01, 1.8722e-03, 7.2769e-03, 2.7461e-01, 2.7461e-01, 7.2769e-03,
        1.8722e-03, 7.2769e-03, 1.8722e-03, 1.2839e-04, 4.5020e-04, 7.5969e-04,
        4.5020e-04, 1.2839e-04, 9.7802e-03, 4.5020e-04, 4.5020e-04, 9.7802e-03,
        1.2562e-04, 7.5969e-04, 9.7802e-03, 8.7077e-02, 7.5477e-02, 4.5020e-04,
        8.7077e-02, 9.7802e-03, 7.5969e-04, 1.2562e-04, 1.7459e+00, 7.5478e-02,
        4.5020e-04, 7.5477e-02, 1.2839e-04, 9.7802e-03, 8.7077e-02, 1.7459e+00,
        1.7459e+00, 9.7802e-03, 1.2562e-04, 2.7461e-01, 1.8722e-03, 7.2769e-03,
        2.7461e-01, 2.7461e-01, 7.2769e-

In [24]:
expGamma2, expGamma_onsite2 = calculate_expgamma(highest_occu_Us=skp.skdict['Highest_Occu_U'], 
                    edge_lengths=data[AtomicDataDict.EDGE_LENGTH_KEY],
                    edge_atom_types=edge_atom_types,
                    node_atom_types=data[AtomicDataDict.ATOM_TYPE_KEY])

In [25]:
max([idp_sk.orbpair_maps[i].stop for i in idp_sk.orbpair_maps.keys()])

4